# DatasetMain Commitment Juncture Paper Tables

This notebook builds paper-ready commitment-prevalence tables directly from the raw DatasetMain localization JSONs.

Definitions:
- Deceptive / truthful example: explicit `deceptive` label from each bundle's `examples.jsonl`, with localization-internal labels used only as a fallback when the example record is missing.
- Deceptive commitment: `delta_deception_rate > 0.3`
- Truthful commitment: `delta_deception_rate < -0.3`
- Commitment example location: the first qualifying commitment sentence as a fraction of the full reasoning-trace length.
  - `Model x Environment`: `mean [bootstrap 95% CI]`
  - `Model` pooled across environments: `mean [bootstrap 95% CI]`

The notebook caches parsed example summaries under `ARTIFACT_DIR` so future runs can skip the expensive raw-JSON scan. Set `FORCE_REBUILD_SUMMARIES = True` to rebuild the cache from scratch.


In [4]:
from __future__ import annotations

from pathlib import Path
import importlib
import json

import pandas as pd
from IPython.display import Markdown, display

import datasetmain_commitment_juncture_prevalence_lib as cj


cj = importlib.reload(cj)

DATASETMAIN_ROOT = cj.DATASETMAIN_ROOT
DELTA_THRESHOLD = cj.DELTA_DECEPTION_THRESHOLD
MAX_JSON_FILES_PER_BUNDLE = None
BOOTSTRAP_NUM_RESAMPLES = cj.BOOTSTRAP_NUM_RESAMPLES
SHOW_PROGRESS = True
PROGRESS_LEVEL = 'bundle'
SAVE_ARTIFACTS = True
LOAD_SUMMARY_CACHE = True
SAVE_SUMMARY_CACHE = True
FORCE_REBUILD_SUMMARIES = False
ARTIFACT_DIR = Path('/playpen-ssd/smerrill/deception2/dataset_scripts/outputs/datasetmain_commitment_juncture_prevalence_paper')
SUMMARY_CACHE_VERSION = 'prevalence_example_cache_v2'

pd.options.display.max_columns = 200


def md(text: str) -> None:
    display(Markdown(text))


def ensure_artifact_dir() -> Path:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    return ARTIFACT_DIR


def maybe_save_table(df: pd.DataFrame, stem: str) -> None:
    if not SAVE_ARTIFACTS:
        return
    out_dir = ensure_artifact_dir()
    df.to_csv(out_dir / f'{stem}.csv', index=False)


def _escape_markdown_cell(value: object) -> str:
    text = '' if value is None else str(value)
    return text.replace('|', '&#124;').replace(chr(10), '<br>')


def _render_markdown_table(
    df: pd.DataFrame,
    *,
    formatters: dict[str, str] | None = None,
) -> None:
    if df.empty:
        md('_No rows available._')
        return

    rendered_df = df.copy()
    for column, fmt in (formatters or {}).items():
        if column not in rendered_df.columns:
            continue
        formatted_values = []
        for value in rendered_df[column].tolist():
            if pd.isna(value):
                formatted_values.append('')
            else:
                formatted_values.append(fmt.format(value))
        rendered_df[column] = formatted_values

    headers = [str(column) for column in rendered_df.columns]
    lines = [
        '| ' + ' | '.join(_escape_markdown_cell(header) for header in headers) + ' |',
        '| ' + ' | '.join('---' for _ in headers) + ' |',
    ]
    for row in rendered_df.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(_escape_markdown_cell(value) for value in row) + ' |')
    display(Markdown(chr(10).join(lines)))


def display_paper_table(df: pd.DataFrame) -> None:
    formatters = {}
    for column in df.columns:
        if column.endswith('Examples'):
            formatters[column] = '{:,.0f}'
        elif column.endswith('Fraction'):
            formatters[column] = '{:.1%}'
    _render_markdown_table(df, formatters=formatters)


def summary_cache_paths() -> dict[str, Path]:
    out_dir = ensure_artifact_dir()
    return {
        'metadata': out_dir / 'summary_cache_metadata.json',
        'inventory': out_dir / 'inventory_df.pkl',
        'example': out_dir / 'example_df.pkl',
        'parse_error': out_dir / 'parse_error_df.pkl',
    }


def build_summary_cache_metadata() -> dict[str, object]:
    return {
        'cache_version': SUMMARY_CACHE_VERSION,
        'dataset_root': str(DATASETMAIN_ROOT),
        'delta_threshold': float(DELTA_THRESHOLD),
        'max_json_files_per_bundle': MAX_JSON_FILES_PER_BUNDLE,
    }


def _metadata_without_dataset_root(metadata: dict[str, object]) -> dict[str, object]:
    return {key: value for key, value in metadata.items() if key != 'dataset_root'}


def has_complete_summary_cache() -> bool:
    paths = summary_cache_paths()
    if not all(path.exists() for path in paths.values()):
        return False
    try:
        metadata = json.loads(paths['metadata'].read_text(encoding='utf-8'))
    except Exception:
        return False
    expected_metadata = build_summary_cache_metadata()
    return _metadata_without_dataset_root(metadata) == _metadata_without_dataset_root(expected_metadata)


def load_summary_cache() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths = summary_cache_paths()
    inventory_df = pd.read_pickle(paths['inventory'])
    example_df = pd.read_pickle(paths['example'])
    parse_error_df = pd.read_pickle(paths['parse_error'])
    return inventory_df, example_df, parse_error_df


def save_summary_cache(
    inventory_df: pd.DataFrame,
    example_df: pd.DataFrame,
    parse_error_df: pd.DataFrame,
) -> None:
    if not SAVE_SUMMARY_CACHE:
        return
    paths = summary_cache_paths()
    inventory_df.to_pickle(paths['inventory'])
    example_df.to_pickle(paths['example'])
    parse_error_df.to_pickle(paths['parse_error'])
    paths['metadata'].write_text(
        json.dumps(build_summary_cache_metadata(), indent=2, sort_keys=True),
        encoding='utf-8',
    )


In [5]:
if not hasattr(cj, 'load_datasetmain_localization_example_df'):
    cj = importlib.reload(cj)

summary_source = 'raw_json'
if LOAD_SUMMARY_CACHE and (not FORCE_REBUILD_SUMMARIES) and has_complete_summary_cache():
    inventory_df, example_df, parse_error_df = load_summary_cache()
    summary_source = 'cache'
else:
    inventory_df, example_df, parse_error_df = cj.load_datasetmain_localization_example_df(
        DATASETMAIN_ROOT,
        max_json_files_per_bundle=MAX_JSON_FILES_PER_BUNDLE,
        show_progress=SHOW_PROGRESS,
        progress_level=PROGRESS_LEVEL,
    )
    save_summary_cache(inventory_df, example_df, parse_error_df)

coverage_table_df = inventory_df.loc[
    :,
    [
        'model_display',
        'env_display',
        'json_file_count',
        'loaded_examples',
        'usable_examples',
        'unusable_examples',
    ],
].rename(
    columns={
        'model_display': 'Model',
        'env_display': 'Environment',
        'json_file_count': 'Localization JSONs',
        'loaded_examples': 'Summarized Examples',
        'usable_examples': 'Usable Examples',
        'unusable_examples': 'Unusable Examples',
    }
)

md('## Localization Coverage')
_render_markdown_table(
    coverage_table_df,
    formatters={
        'Localization JSONs': '{:,.0f}',
        'Summarized Examples': '{:,.0f}',
        'Usable Examples': '{:,.0f}',
        'Unusable Examples': '{:,.0f}',
    },
)

memory_mib = example_df.memory_usage(deep=True).sum() / (1024 ** 2) if not example_df.empty else 0.0
source_text = 'cached summary pickles' if summary_source == 'cache' else 'raw localization JSONs'
md(
    f'Loaded `{len(example_df):,}` example summaries from {source_text}. '
    f'Parse errors: `{len(parse_error_df):,}`. Example summary frame memory: `{memory_mib:.1f} MiB`. '
    f'Progress settings: `SHOW_PROGRESS={SHOW_PROGRESS}`, `PROGRESS_LEVEL={PROGRESS_LEVEL}`. '
    f'Cache status: `summary_source={summary_source}`, `FORCE_REBUILD_SUMMARIES={FORCE_REBUILD_SUMMARIES}`.'
)

maybe_save_table(coverage_table_df, 'json_localization_coverage')
maybe_save_table(parse_error_df, 'parse_errors')


## Localization Coverage

| Model | Environment | Localization JSONs | Summarized Examples | Usable Examples | Unusable Examples |
| --- | --- | --- | --- | --- | --- |
| GPT-OSS-20B | AdvisorAudit | 5,000 | 5,000 | 5,000 | 0 |
| GPT-OSS-20B | BS | 5,000 | 5,000 | 5,000 | 0 |
| GPT-OSS-20B | CarSales | 5,000 | 5,000 | 5,000 | 0 |
| GPT-OSS-20B | Gridworld | 5,000 | 5,000 | 5,000 | 0 |
| GPT-OSS-20B | Interview | 5,000 | 5,000 | 5,000 | 0 |
| Llama-8B | AdvisorAudit | 5,000 | 5,000 | 5,000 | 0 |
| Llama-8B | BS | 5,000 | 5,000 | 5,000 | 0 |
| Llama-8B | CarSales | 5,000 | 5,000 | 5,000 | 0 |
| Llama-8B | Gridworld | 5,000 | 5,000 | 5,000 | 0 |
| Llama-8B | Interview | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-14B | AdvisorAudit | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-14B | BS | 5,000 | 4,999 | 4,999 | 0 |
| Qwen-14B | CarSales | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-14B | Gridworld | 5,000 | 4,999 | 4,999 | 0 |
| Qwen-14B | Interview | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-7B | AdvisorAudit | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-7B | BS | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-7B | CarSales | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-7B | Gridworld | 5,000 | 5,000 | 5,000 | 0 |
| Qwen-7B | Interview | 5,000 | 5,000 | 5,000 | 0 |

Loaded `99,998` example summaries from cached summary pickles. Parse errors: `2`. Example summary frame memory: `40.9 MiB`. Progress settings: `SHOW_PROGRESS=True`, `PROGRESS_LEVEL=bundle`. Cache status: `summary_source=cache`, `FORCE_REBUILD_SUMMARIES=False`.

In [6]:
env_model_stats_df = cj.build_commitment_example_statistics(
    example_df,
    ['model_display', 'env_display'],
    bootstrap_location_ci=True,
    bootstrap_num_resamples=BOOTSTRAP_NUM_RESAMPLES,
)
model_stats_df = cj.build_commitment_example_statistics(
    example_df,
    ['model_display'],
    bootstrap_location_ci=True,
    bootstrap_num_resamples=BOOTSTRAP_NUM_RESAMPLES,
)

paper_env_model_table_df = cj.make_commitment_fraction_location_table(
    env_model_stats_df,
    location_interval_style='bootstrap_ci',
)
paper_model_table_df = cj.make_commitment_fraction_location_table(
    model_stats_df,
    location_interval_style='bootstrap_ci',
)

md('## Model Pooled Across Environments')
display_paper_table(paper_model_table_df)

md('## Model x Environment')
display_paper_table(paper_env_model_table_df)

maybe_save_table(env_model_stats_df, 'env_model_stats_raw')
maybe_save_table(model_stats_df, 'model_stats_raw')
maybe_save_table(paper_env_model_table_df, 'env_model_paper_table')
maybe_save_table(paper_model_table_df, 'model_paper_table')


## Model Pooled Across Environments

| Model | Deceptive Examples | Deceptive Commitment Fraction | Deceptive Commitment Location | Truthful Examples | Truthful Commitment Fraction | Truthful Commitment Location |
| --- | --- | --- | --- | --- | --- | --- |
| GPT-OSS-20B | 12,500 | 16.0% | 57.3% [56.1%, 58.6%] | 12,500 | 71.0% | 52.5% [51.9%, 53.1%] |
| Llama-8B | 12,500 | 40.6% | 58.4% [57.7%, 59.2%] | 12,500 | 34.1% | 66.8% [66.0%, 67.6%] |
| Qwen-7B | 12,508 | 58.2% | 66.5% [66.0%, 67.0%] | 12,492 | 21.2% | 67.0% [66.2%, 67.8%] |
| Qwen-14B | 12,499 | 26.1% | 65.7% [64.7%, 66.6%] | 12,499 | 36.5% | 65.9% [65.2%, 66.6%] |

## Model x Environment

| Model | Environment | Deceptive Examples | Deceptive Commitment Fraction | Deceptive Commitment Location | Truthful Examples | Truthful Commitment Fraction | Truthful Commitment Location |
| --- | --- | --- | --- | --- | --- | --- | --- |
| GPT-OSS-20B | AdvisorAudit | 2,500 | 2.2% | 54.1% [48.2%, 60.8%] | 2,500 | 78.6% | 68.1% [67.1%, 69.1%] |
| GPT-OSS-20B | BS | 2,500 | 39.1% | 54.6% [52.9%, 56.4%] | 2,500 | 49.0% | 66.1% [64.7%, 67.5%] |
| GPT-OSS-20B | CarSales | 2,500 | 14.5% | 47.5% [45.3%, 49.8%] | 2,500 | 80.1% | 33.5% [32.8%, 34.3%] |
| GPT-OSS-20B | Gridworld | 2,500 | 17.2% | 66.3% [63.6%, 69.0%] | 2,500 | 65.7% | 44.7% [43.4%, 46.1%] |
| GPT-OSS-20B | Interview | 2,500 | 7.1% | 71.6% [68.0%, 75.2%] | 2,500 | 81.6% | 54.0% [52.8%, 55.2%] |
| Llama-8B | AdvisorAudit | 2,500 | 31.4% | 59.2% [57.1%, 61.1%] | 2,500 | 29.6% | 75.1% [73.8%, 76.5%] |
| Llama-8B | BS | 2,500 | 38.6% | 73.0% [71.5%, 74.7%] | 2,500 | 12.7% | 75.5% [72.9%, 78.3%] |
| Llama-8B | CarSales | 2,500 | 78.3% | 56.5% [55.4%, 57.8%] | 2,500 | 43.9% | 44.7% [43.1%, 46.3%] |
| Llama-8B | Gridworld | 2,500 | 50.4% | 48.3% [47.1%, 49.8%] | 2,500 | 13.4% | 67.2% [64.6%, 69.9%] |
| Llama-8B | Interview | 2,500 | 4.1% | 73.6% [69.5%, 77.4%] | 2,500 | 71.0% | 75.3% [74.3%, 76.4%] |
| Qwen-7B | AdvisorAudit | 2,500 | 64.5% | 64.8% [63.8%, 65.8%] | 2,500 | 12.6% | 71.5% [69.5%, 73.5%] |
| Qwen-7B | BS | 2,500 | 44.4% | 67.2% [65.4%, 68.8%] | 2,500 | 18.5% | 73.9% [72.1%, 75.9%] |
| Qwen-7B | CarSales | 2,500 | 42.2% | 61.1% [59.9%, 62.5%] | 2,500 | 42.0% | 61.0% [59.7%, 62.2%] |
| Qwen-7B | Gridworld | 2,500 | 81.8% | 67.0% [66.2%, 67.9%] | 2,500 | 1.6% | 72.5% [67.3%, 77.6%] |
| Qwen-7B | Interview | 2,508 | 57.9% | 70.9% [69.8%, 72.0%] | 2,492 | 31.5% | 68.9% [67.5%, 70.4%] |
| Qwen-14B | AdvisorAudit | 2,500 | 26.0% | 58.6% [56.8%, 60.4%] | 2,500 | 57.7% | 62.9% [61.8%, 64.0%] |
| Qwen-14B | BS | 2,499 | 26.1% | 62.6% [60.4%, 64.7%] | 2,500 | 10.9% | 73.3% [70.7%, 76.0%] |
| Qwen-14B | CarSales | 2,500 | 27.7% | 56.3% [54.5%, 58.1%] | 2,500 | 48.1% | 62.1% [60.7%, 63.6%] |
| Qwen-14B | Gridworld | 2,500 | 48.6% | 75.8% [74.3%, 77.3%] | 2,499 | 2.3% | 65.5% [57.2%, 73.0%] |
| Qwen-14B | Interview | 2,500 | 2.0% | 80.7% [74.0%, 87.6%] | 2,500 | 63.7% | 70.2% [68.9%, 71.4%] |